In [ ]:
# Esta celda muestra qué intérprete de Python está usando el notebook.
import sys
print(sys.executable)

d:\ProyectoIA\.venv\Scripts\python.exe


In [ ]:
# Esta celda instala la librería TensorFlow si aún no está disponible.
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Esta celda importa las librerías necesarias y carga las imágenes de entrenamiento y prueba.
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

#Rutas actualizadas para entrar primero a la carpeta 'archive'
ruta_train = 'train'
ruta_test = 'test'

# Cargamos las imágenes de entrenamiento
train_ds = tf.keras.utils.image_dataset_from_directory(
    ruta_train,
    color_mode='grayscale',
    image_size=(48, 48),
    batch_size=32
)

# Cargamos las imágenes de prueba para validar qué tan bien aprende la IA
test_ds = tf.keras.utils.image_dataset_from_directory(
    ruta_test,
    color_mode='grayscale',
    image_size=(48, 48),
    batch_size=32
)

# Guardamos los nombres de las emociones (clases)
nombres_clases = train_ds.class_names
print(f"Las emociones que tu IA va a reconocer son: {nombres_clases}")

Found 28709 files belonging to 7 classes.
Found 3589 files belonging to 7 classes.
Las emociones que tu IA va a reconocer son: ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']


In [ ]:
# Esta celda define la red neuronal que va a aprender a reconocer emociones.
modelo = models.Sequential([
    layers.Input(shape=(48, 48, 1)),
    layers.Rescaling(1./255), # Normalización de píxeles
    
    # Capas Convolucionales: detectan rasgos como ojos, boca o cejas
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Capas de Clasificación: deciden qué emoción es
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Ayuda a que no se "memorice" las fotos
    layers.Dense(len(nombres_clases), activation='softmax')
])

# Configuramos cómo va a aprender la IA
modelo.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Imprimimos el resumen del modelo para ver su estructura
modelo.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 48, 48, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 355,847 (1.36 MB)

 Trainable params: 355,847 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Esta celda entrena el modelo con los datos y luego lo guarda en un archivo.
print("Entrenando la IA... esto puede tardar unos minutos.")
historial = modelo.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15
)

# Guardamos el modelo en un archivo para usarlo después en la aplicación web
modelo.save('modelo_emociones.keras')
print("¡Exito! El modelo se ha guardado como 'modelo_emociones.keras'")

Entrenando la IA... esto puede tardar unos minutos.
Epoch 1/15
898/898 ━━━━━━━━━━━━━━━━━━━━ 24s 24ms/step - accuracy: 0.2882 - loss: 1.7401 - val_accuracy: 0.3940 - val_loss: 1.5603
Epoch 2/15
898/898 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - accuracy: 0.4088 - loss: 1.5249 - val_accuracy: 0.4550 - val_loss: 1.4226
Epoch 3/15
898/898 ━━━━━━━━━━━━━━━━━━━━ 21s 23ms/step - accuracy: 0.4565 - loss: 1.4146 - val_accuracy: 0.4896 - val_loss: 1.3358
Epoch 4/15
898/898 ━━━━━━━━━━━━━━━━━━━━ 21s 23ms/step - accuracy: 0.4875 - loss: 1.3412 - val_accuracy: 0.5077 - val_loss: 1.2816
Epoch 5/15
898/898 ━━━━━━━━━━━━━━━━━━━━ 21s 23ms/step - accuracy: 0.5112 - loss: 1.2871 - val_accuracy: 0.5188 - val_loss: 1.2444
Epoch 6/15
898/898 ━━━━━━━━━━━━━━━━━━━━ 20s 23ms/step - accuracy: 0.5334 - loss: 1.2370 - val_accuracy: 0.5194 - val_loss: 1.2442
Epoch 7/15
898/898 ━━━━━━━━━━━━━━━━━━━━ 21s 23ms/step - accuracy: 0.5467 - loss: 1.2037 - val_accuracy: 0.5330 - val_loss: 1.2170
Epoch 8/15
898/898 ━━━━━━━━━━━━━━━━━━━